In [34]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import joblib

In [ ]:
car_data = pd.read_csv("datasets/car_price_prediction.csv")

In [ ]:
car_data['Turbo'] = car_data['Engine volume'].str.contains('Turbo').astype(int)

car_data['Engine volume'] = (
    car_data['Engine volume']
    .str.replace(' Turbo', '')
    .astype(float)
)

car_data['Mileage'] = (
    car_data['Mileage']
    .str.replace(' km', '')
    .astype(int)
)

car_data['Levy'] = car_data['Levy'].replace('-', np.nan)
car_data['Levy'] = pd.to_numeric(car_data['Levy'])

In [ ]:
X = car_data.drop(['Price', 'ID'], axis=1)
y = car_data['Price']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

In [ ]:
model_counts = X_train['Model'].value_counts()

rare_models = model_counts[model_counts < 10].index

X_train['Model'] = X_train['Model'].replace(rare_models, 'Other')
X_test['Model'] = X_test['Model'].replace(rare_models, 'Other')

In [ ]:
numeric_features = X_train.select_dtypes(include='number').columns
categorical_features = X_train.select_dtypes(exclude='number').columns

In [ ]:
numerical_processor = SimpleImputer(strategy='median')

categorical_processor = OneHotEncoder(handle_unknown='ignore')

In [ ]:
preprocessor = ColumnTransformer([
    ("numeric", numerical_processor, numeric_features),
    ("categorical", categorical_processor, categorical_features)
])

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

In [ ]:
print(X_train_processed.shape)
print(X_test_processed.shape)

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(X_train_processed, y_train)

In [ ]:
y_pred = model.predict(X_test_processed)

In [ ]:
print(y_pred[:10])

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

In [ ]:
print(car_data['Price'].describe())

In [ ]:
print(car_data.nlargest(20, 'Price')[['Price', 'Manufacturer', 'Model', 'Prod. year']])

In [ ]:
print(car_data.nsmallest(20, 'Price')[['Price', 'Manufacturer', 'Model', 'Prod. year']])

In [ ]:
price_z = (
    (car_data['Price'] - car_data['Price'].mean())
    / car_data['Price'].std()
)

outliers = car_data[price_z.abs() > 3]

print("Number of potential outliers:", len(outliers))
print(outliers[['Price', 'Manufacturer', 'Model', 'Prod. year']])

In [ ]:
print(16983 in X_train.index)
print(16983 in X_test.index)

In [ ]:
X_train = X_train.drop(index=16983)
y_train = y_train.drop(index=16983)

In [ ]:
print(X_train.shape)
print(y_train.shape)

In [ ]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [ ]:
model.fit(X_train_processed, y_train)

In [ ]:
y_pred = model.predict(X_test_processed)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R²:", r2)

In [36]:
joblib.dump(model, "model.pkl")
joblib.dump(preprocessor, "preprocessor.pkl")

['preprocessor.pkl']